# Feature Engineering with Snowflake Feature Store

This notebook demonstrates Snowflake's **Feature Store** capabilities:

- **Entities**: Define join keys for feature lookup
- **Feature Views**: Collections of features backed by Dynamic Tables with incremental refresh
- **Dataset Generation**: Create versioned training datasets from the Feature Store

Feature Views are automatically maintained by Snowflake's Dynamic Tables, ensuring features
are always fresh without manual pipeline management.

In [ ]:
import pandas as pd
import numpy as np

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, avg, count, sum as sum_, stddev, datediff, lag, hour, dayofweek,
    when, lit, max as max_, min as min_, abs as abs_, count_distinct, sql_expr
)
from snowflake.snowpark.types import IntegerType, FloatType
from snowflake.snowpark import Window

from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

session = get_active_session()
session.query_tag = {"origin": "ml_deep_dive", "name": "banking_fraud_detection", "notebook": "01_feature_engineering"}

DB = "BANKING_ML_DEMO"
SCHEMA = "FRAUD_DETECTION"
WH = "ML_DEMO_WH"
VERSION = "V1"

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(WH)

print(f"Context: {session.get_fully_qualified_current_schema()}")

## Initialize Feature Store

The Feature Store is backed by a Snowflake schema. Feature Views are materialized as Dynamic Tables.

In [ ]:
fs = FeatureStore(
    session=session,
    database=DB,
    name=SCHEMA,
    default_warehouse=WH,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print("Feature Store initialized")
print(f"Existing feature views: {len(fs.list_feature_views().collect())}")
print(f"Existing entities: {len(fs.list_entities().collect())}")

## Register Entities

Entities define the join keys used for feature lookup at training and inference time.

In [ ]:
# Customer Entity
try:
    customer_entity = fs.get_entity("CUSTOMER_ENTITY")
    print("Retrieved existing CUSTOMER_ENTITY")
except:
    customer_entity = Entity(
        name="CUSTOMER_ENTITY",
        join_keys=["CUSTOMER_ID"],
        desc="Customer entity for fraud detection features"
    )
    fs.register_entity(customer_entity)
    print("Registered new CUSTOMER_ENTITY")

# Merchant Entity
try:
    merchant_entity = fs.get_entity("MERCHANT_ENTITY")
    print("Retrieved existing MERCHANT_ENTITY")
except:
    merchant_entity = Entity(
        name="MERCHANT_ENTITY",
        join_keys=["MERCHANT_ID"],
        desc="Merchant entity for fraud detection features"
    )
    fs.register_entity(merchant_entity)
    print("Registered new MERCHANT_ENTITY")

# List all entities
fs.list_entities().show()

## Create Customer Features

Customer-level aggregations that capture spending behavior and patterns.

In [ ]:
raw_txns = session.table("RAW_TRANSACTIONS")

# Customer-level feature aggregations
customer_features_df = raw_txns.group_by("CUSTOMER_ID").agg(
    avg("AMOUNT").alias("CUST_AVG_TRANSACTION_AMT"),
    count("*").alias("CUST_TOTAL_TRANSACTIONS"),
    stddev("AMOUNT").alias("CUST_STDDEV_AMOUNT"),
    max_("AMOUNT").alias("CUST_MAX_AMOUNT"),
    min_("AMOUNT").alias("CUST_MIN_AMOUNT"),
    count_distinct("MERCHANT_ID").alias("CUST_UNIQUE_MERCHANTS"),
    avg(col("IS_ONLINE").cast(FloatType())).alias("CUST_PCT_ONLINE"),
    avg(hour("TIMESTAMP").cast(FloatType())).alias("CUST_AVG_TRANSACTION_HOUR"),
    count_distinct("MERCHANT_CATEGORY").alias("CUST_UNIQUE_CATEGORIES"),
    avg("CUSTOMER_AGE").alias("CUST_AGE"),
    avg("CUSTOMER_INCOME").alias("CUST_INCOME")
)

print("Customer features schema:")
for field in customer_features_df.schema.fields:
    print(f"  {field.name}: {field.datatype}")
customer_features_df.show(5)

In [ ]:
# Register Customer Feature View
customer_feature_descriptions = {
    "CUST_AVG_TRANSACTION_AMT": "Average transaction amount per customer",
    "CUST_TOTAL_TRANSACTIONS": "Total number of transactions",
    "CUST_STDDEV_AMOUNT": "Standard deviation of transaction amounts",
    "CUST_MAX_AMOUNT": "Maximum single transaction amount",
    "CUST_MIN_AMOUNT": "Minimum single transaction amount",
    "CUST_UNIQUE_MERCHANTS": "Number of distinct merchants visited",
    "CUST_PCT_ONLINE": "Percentage of online transactions",
    "CUST_AVG_TRANSACTION_HOUR": "Average hour of day for transactions",
    "CUST_UNIQUE_CATEGORIES": "Number of distinct merchant categories",
    "CUST_AGE": "Customer age",
    "CUST_INCOME": "Customer annual income"
}

try:
    cust_fv = fs.get_feature_view("CUSTOMER_FEATURES_FV", VERSION)
    print(f"Retrieved existing CUSTOMER_FEATURES_FV/{VERSION}")
except:
    cust_fv = FeatureView(
        name="CUSTOMER_FEATURES_FV",
        entities=[customer_entity],
        feature_df=customer_features_df,
        refresh_freq="1 day",
        desc="Customer-level behavioral features for fraud detection"
    ).attach_feature_desc(customer_feature_descriptions)
    
    cust_fv = fs.register_feature_view(
        feature_view=cust_fv,
        version=VERSION,
        block=True
    )
    print(f"Registered CUSTOMER_FEATURES_FV/{VERSION}")

# Show feature view details
fs.list_feature_views().show()

## Create Merchant Features

Merchant-level aggregations capturing transaction patterns and fraud rates.

In [ ]:
# Merchant-level feature aggregations
merchant_features_df = raw_txns.group_by("MERCHANT_ID").agg(
    avg("AMOUNT").alias("MERCH_AVG_TRANSACTION_AMT"),
    count("*").alias("MERCH_TOTAL_TRANSACTIONS"),
    count_distinct("CUSTOMER_ID").alias("MERCH_UNIQUE_CUSTOMERS"),
    avg(col("IS_FRAUD").cast(FloatType())).alias("MERCH_FRAUD_RATE"),
    stddev("AMOUNT").alias("MERCH_STDDEV_AMOUNT"),
    max_("AMOUNT").alias("MERCH_MAX_AMOUNT")
)

print("Merchant features:")
merchant_features_df.show(5)

In [ ]:
# Register Merchant Feature View
merchant_feature_descriptions = {
    "MERCH_AVG_TRANSACTION_AMT": "Average transaction amount at this merchant",
    "MERCH_TOTAL_TRANSACTIONS": "Total transactions at this merchant",
    "MERCH_UNIQUE_CUSTOMERS": "Number of unique customers",
    "MERCH_FRAUD_RATE": "Historical fraud rate at this merchant",
    "MERCH_STDDEV_AMOUNT": "Standard deviation of transaction amounts",
    "MERCH_MAX_AMOUNT": "Maximum single transaction amount"
}

try:
    merch_fv = fs.get_feature_view("MERCHANT_FEATURES_FV", VERSION)
    print(f"Retrieved existing MERCHANT_FEATURES_FV/{VERSION}")
except:
    merch_fv = FeatureView(
        name="MERCHANT_FEATURES_FV",
        entities=[merchant_entity],
        feature_df=merchant_features_df,
        refresh_freq="1 day",
        desc="Merchant-level features for fraud detection"
    ).attach_feature_desc(merchant_feature_descriptions)
    
    merch_fv = fs.register_feature_view(
        feature_view=merch_fv,
        version=VERSION,
        block=True
    )
    print(f"Registered MERCHANT_FEATURES_FV/{VERSION}")

fs.list_feature_views().show()

## Create Transaction-Level Features

Transaction-level features derived from temporal and behavioral patterns.

In [ ]:
# Transaction-level feature engineering using window functions
cust_window = Window.partition_by("CUSTOMER_ID")

txn_features_df = raw_txns.select(
    col("TRANSACTION_ID"),
    col("CUSTOMER_ID"),
    col("MERCHANT_ID"),
    col("AMOUNT"),
    col("TIMESTAMP"),
    col("IS_ONLINE"),
    col("IS_FRAUD"),
    # Temporal features
    hour("TIMESTAMP").alias("TXN_HOUR"),
    dayofweek("TIMESTAMP").alias("TXN_DAY_OF_WEEK"),
    when(dayofweek("TIMESTAMP").isin([0, 6]), 1).otherwise(0).alias("TXN_IS_WEEKEND"),
    # Late night flag (10pm - 5am)
    when((hour("TIMESTAMP") >= 22) | (hour("TIMESTAMP") <= 5), 1).otherwise(0).alias("TXN_IS_LATE_NIGHT"),
    # Amount relative to customer average
    (col("AMOUNT") / avg("AMOUNT").over(cust_window)).alias("TXN_AMOUNT_RATIO"),
    # Flag: amount > 3x customer average
    when(col("AMOUNT") > lit(3) * avg("AMOUNT").over(cust_window), 1).otherwise(0).alias("TXN_IS_HIGH_AMOUNT"),
)

print("Transaction-level features:")
txn_features_df.show(5)

In [ ]:
# Save transaction features as a table for training
txn_features_df.write.mode("overwrite").save_as_table("TXN_FEATURES")
print(f"Saved {session.table('TXN_FEATURES').count():,} rows to TXN_FEATURES")

## Generate Training Dataset

Use the Feature Store to generate a versioned dataset by joining features from our Feature Views
onto a spine DataFrame.

In [ ]:
# Create spine DataFrame with entity keys and labels
spine_df = session.table("TXN_FEATURES").select(
    "TRANSACTION_ID", "CUSTOMER_ID", "MERCHANT_ID", "TIMESTAMP",
    "AMOUNT", "IS_ONLINE", "IS_FRAUD",
    "TXN_HOUR", "TXN_DAY_OF_WEEK", "TXN_IS_WEEKEND",
    "TXN_IS_LATE_NIGHT", "TXN_AMOUNT_RATIO", "TXN_IS_HIGH_AMOUNT"
)

# Generate dataset enriched with Feature Store features
dataset = fs.generate_dataset(
    name=f"FRAUD_TRAINING_DATASET_{VERSION}",
    spine_df=spine_df,
    features=[cust_fv, merch_fv],
    spine_label_cols=["IS_FRAUD"]
)

# Read and inspect the generated dataset
dataset_df = dataset.read.to_snowpark_dataframe()
print(f"Training dataset: {dataset_df.count():,} rows, {len(dataset_df.columns)} columns")
print(f"\nColumns: {dataset_df.columns}")
dataset_df.show(5)

In [ ]:
# Dataset statistics
dataset_pd = dataset_df.to_pandas()
print(f"Dataset shape: {dataset_pd.shape}")
print(f"\nFraud distribution:")
print(dataset_pd["IS_FRAUD"].value_counts())
print(f"\nFraud rate: {dataset_pd['IS_FRAUD'].mean():.2%}")
print(f"\nFeature summary:")
print(dataset_pd.describe().round(2))

## View in Snowsight

Navigate to **Snowsight > AI & ML > Feature Store** to explore:
- Registered Entities and their join keys
- Feature Views with descriptions, versions, and refresh schedules
- Dynamic Table status and refresh history
- Generated Datasets with lineage back to Feature Views

### Next Steps

Proceed to **`02_experiment_tracking.ipynb`** to train models and track experiments.